# Chapter 05 — Transformer Block

The Transformer is the architecture behind every modern large language model — GPT-4, Claude, Llama, Gemini. In this chapter, you will assemble the complete Transformer decoder block by block, understanding not just what each component does, but why it's necessary. By the end, you'll build GPT-2 with 124 million parameters.

### Glossary / Glossário

• transformer — neural architecture based on self-attention, introduced in "Attention Is All You Need" (2017). / Arquitetura neural baseada em self-attention, introduzida em "Attention Is All You Need" (2017).
• residual connection — shortcut that adds a layer's input to its output, creating a gradient highway. / Atalho que soma a entrada de uma camada à sua saída, criando autoestrada de gradientes.
• LayerNorm — Layer Normalization, normalizes activations across features for each example. / Normalização de camada, normaliza ativações ao longo das features para cada exemplo.
• FFN — Feed-Forward Network, two linear layers with activation between them in each transformer block. / Rede Feed-Forward, duas camadas lineares com ativação entre elas em cada bloco transformer.
• decoder-only — transformer variant using only masked self-attention, standard for language models like GPT. / Variante do transformer que usa apenas self-attention mascarada, padrão para modelos de linguagem como GPT.

In [ ]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        # d_model: model dimension — width of the representation at each position
        # n_heads: number of attention heads — each learns different relationship patterns
        # d_ff: FFN inner dimension — typically 4× d_model for capacity
        self.ln1 = nn.LayerNorm(d_model)   # ln1: layer normalization — stabilizes training by normalizing activations
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)  # attn: multi-head self-attention — lets tokens communicate with each other
        self.ln2 = nn.LayerNorm(d_model)   # ln2: layer normalization — stabilizes training by normalizing activations
        self.ffn = nn.Sequential(          # ffn: feed-forward network — processes each position independently (4x expansion)
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        # Pre-LayerNorm + Residual Connection
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=mask)
        x = x + attn_out  # x + attn_out = residual connection — gradient highway preventing vanishing gradients

        normed = self.ln2(x)
        x = x + self.ffn(normed)  # residual connection again
        return x

# Demo: forward pass
block = TransformerBlock(d_model=64, n_heads=4, d_ff=256)
x = torch.randn(2, 10, 64)  # batch=2, seq=10, d_model=64
out = block(x)
params = sum(p.numel() for p in block.parameters())
print(f"Input shape:  {tuple(x.shape)}")
print(f"Output shape: {tuple(out.shape)}")
print(f"Parameters:   {params:,}")

# === Aha: training a transformer block on a simple task ===
import torch.nn.functional as F
linear_head = torch.nn.Linear(64, 64)
optimizer = torch.optim.Adam(list(block.parameters()) + list(linear_head.parameters()), lr=0.001)
print(f"\n{'Step':>5s} | {'Loss':>8s}")
print("-" * 20)
for step in range(6):
    x = torch.randn(2, 10, 64)
    target = x.clone()  # learn identity mapping
    out = linear_head(block(x))
    loss = F.mse_loss(out, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"{step:>5d} | {loss.item():>8.4f}")
print("\nLoss decreases — the transformer block is learning!")

---

**Course:** Aprenda LLM | **Chapter 05**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.